In [1]:
import pandas as pd
import time
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from langchain_aws import ChatBedrockConverse
from typing import List, Dict

In [2]:
jd = pd.read_excel("data/two_jobs_ads.xlsx")

In [3]:
job_names = []
jd_dict = []
for _, row in jd.iterrows():
    job_names.append(row["Role Title"])
    jd_dict.append(row.to_dict())

In [4]:
personality = list(set(pd.read_excel("data/NEO_PI_R.xlsx")["Facet"]))

In [7]:
from enum import Enum

class Importance(str, Enum):
    NO = "NO"
    NEITHER = "NEITHER"
    YES = "YES"
    
class Evaluation(BaseModel):
    importance: Importance
    rationale: str

class Personality(BaseModel):
    required: Dict[str, Evaluation]

In [8]:
json_parser = JsonOutputParser(pydantic_object=Personality)

In [9]:
with open("prompt/personality_prompt.txt") as f:
    prompt_text = f.read()

In [15]:
format_instructions = json_parser.get_format_instructions()
prompt = PromptTemplate(
    template=prompt_text,
    input_variables=["job_description", "personality"],
    partial_variables={"format_instructions": format_instructions},
)

In [16]:
llm = ChatBedrockConverse(
    model="anthropic.claude-3-5-haiku-20241022-v1:0",
    temperature=0,
    max_tokens=8192,
)

In [17]:
chain = (
    prompt
    | llm
    | json_parser
)

In [13]:
import time

In [14]:
len(jd_dict)

2

In [18]:
required = {}
start_time = time.time()
start_idx = 0
for i in range(len(jd_dict) + 1):
    if (i % 15 == 0 and i != 0) or i == len(jd_dict):
        batch_input = [{
            "job_description": jd_dict[i],
            "personality": personality
        } for i in range(start_idx, i)]
        response = chain.batch(batch_input)
        for j, r in zip(range(start_idx, i), response):
            required[job_names[j]] = r["required"]

        time_lapse = time.time() - start_time
        print(start_idx, i, time_lapse)
        start_idx = i
        if time_lapse < 60 and i != len(jd_dict):
            time.sleep(60 - time_lapse)
        start_time = time.time()

0 2 27.52811312675476


In [19]:
len(required)

2

In [20]:
import json
with open("intermediate_data/personality_likert.json", "w") as f:
    json.dump(required, f)